# IMAGE/EUV Implementation — He+ 30.4 nm Plasmaspheric Imaging / IMAGE/EUV 구현 — He+ 30.4 nm 플라스마권 영상화

**Paper**: Sandel, B. R., Broadfoot, A. L., Curtis, C. C., King, R. A., Stone, T. C., Hill, R. H., Chen, J., Siegmund, O. H. W., Raffanti, R., Allred, D. D., Turley, R. S., and Gallagher, D. L., 2000. The Extreme Ultraviolet Imager Investigation for the IMAGE Mission. *Space Science Reviews* **91**, 197–242. DOI: 10.1023/A:1005263510820.

This notebook reproduces four engineering and science analyses central to EUV / 본 노트북은 EUV의 핵심 공학·과학 분석 네 가지를 재현한다:
1. **Resonant-scattering optical-depth profile / 공명 산란 광학 두께 프로파일** — verifying that the plasmasphere is indeed optically thin at 30.4 nm so that brightness is linear in column density.
2. **Three-head 30° FOV mosaic + Earth-limb registration / 3-헤드 30° FOV 모자이크와 지구 limb 정합** — building the 84°×30° instantaneous fan and the 84°×360° spin-scan annulus, then registering Earth's limb.
3. **Plasmaspheric plume from polar perspective / 극지 시점에서의 플라스마권 plume** — synthesizing a storm-time He+ density distribution and forward-modeling the Rayleigh image.
4. **Toy plasmaspheric tomography / 장난감 플라스마권 단층촬영** — inverting the line-integrated brightness back to a 2-D He+ density map.

All inputs are synthetic and physically motivated by the Sandel et al. (2000) paper. The kernel is `study-with-ai`. / 모든 입력은 Sandel 외 (2000) 논문에 근거한 합성 데이터이며 커널은 `study-with-ai`이다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass
from scipy.optimize import nnls

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11
rng = np.random.default_rng(seed=20000325)  # IMAGE launch date

R_EARTH_KM = 6371.0
RE_TO_CM = R_EARTH_KM * 1.0e5  # 1 R_E in cm

## Part 1: Optical Depth at 30.4 nm / 30.4 nm 광학 두께

The plasmasphere's interpretability hinges on being **optically thin** at 30.4 nm: $\tau \ll 1$ guarantees that observed brightness is linear in He+ column density, removing the need for radiative transfer modeling. We model an equatorial radial profile $n_{\mathrm{He^+}}(L)$ — a flat plasmasphere out to the plasmapause $L_\mathrm{pp}$ followed by a sharp drop — then integrate the line-of-sight optical depth.

플라스마권 영상의 해석 가능성은 30.4 nm에서 **광학적으로 얇음** ($\tau \ll 1$)에 달려 있다. 적도면 He+ 반경 프로파일을 모델링한 뒤 시선 방향 광학 두께를 적분하여 이를 확인한다.

**Key equation / 핵심 식**:
$$ \tau(\mathrm{LOS}) = \sigma_0 \int_{\mathrm{LOS}} n_{\mathrm{He^+}}(s)\,ds, \qquad \sigma_0 \sim 8\times 10^{-13}\,\mathrm{cm}^2 \text{ (line-center)}. $$

In [ ]:
@dataclass
class PlasmasphereModel:
    """Simple equatorial He+ density model.

    Attributes:
        n0_cm3: Peak He+ density at L = 2 (cm^-3).
        L_pp: Plasmapause L-shell location.
        falloff_power: Inner power-law exponent (n ~ L^-falloff_power).
        outer_scale_re: Plasmatrough exponential scale length (R_E).
        n_outer_floor: Plasmatrough floor density (cm^-3).
    """

    n0_cm3: float = 200.0
    L_pp: float = 4.5
    falloff_power: float = 1.0
    outer_scale_re: float = 0.4
    n_outer_floor: float = 1.0

    def density(self, L: np.ndarray) -> np.ndarray:
        """Return He+ density (cm^-3) on the equatorial plane at L-shell L.

        Args:
            L: L-shell value (dimensionless, equivalent to equatorial r/R_E).

        Returns:
            He+ density in cm^-3.
        """
        L = np.asarray(L, dtype=float)
        inner = self.n0_cm3 * np.power(np.maximum(L, 1.0) / 2.0,
                                       -self.falloff_power)
        # Sharp plasmapause: smooth tanh transition centered at L_pp
        weight = 0.5 * (1.0 - np.tanh((L - self.L_pp) / 0.2))
        outer = self.n_outer_floor + (self.n0_cm3 * 0.05) * np.exp(
            -(L - self.L_pp) / self.outer_scale_re)
        outer = np.where(L > self.L_pp, outer, 0.0)
        return inner * weight + outer


def line_integral(density_fn, x_imp_re: float, n_steps: int = 4000,
                  s_max_re: float = 12.0) -> float:
    """Compute equatorial line-of-sight integral N = int n ds (cm^-2).

    The line of sight is parallel to the y-axis at impact parameter x_imp_re.
    The path length is sampled uniformly in s over (-s_max_re, +s_max_re) R_E.

    Args:
        density_fn: Callable mapping L (scalar or array) to density (cm^-3).
        x_imp_re: Impact parameter in R_E.
        n_steps: Number of integration samples.
        s_max_re: Half path length in R_E.

    Returns:
        Column density along the line of sight in cm^-2.
    """
    s = np.linspace(-s_max_re, s_max_re, n_steps)
    L = np.sqrt(x_imp_re ** 2 + s ** 2)
    n = density_fn(L)
    ds_cm = (s[1] - s[0]) * RE_TO_CM
    return float(np.sum(n) * ds_cm)


model = PlasmasphereModel()
L_grid = np.linspace(1.05, 8.0, 400)
n_profile = model.density(L_grid)

fig, ax = plt.subplots()
ax.semilogy(L_grid, n_profile, color='C0', lw=2)
ax.axvline(model.L_pp, color='gray', ls='--',
           label=f'Plasmapause $L_{{pp}}$ = {model.L_pp}')
ax.set_xlabel('L-shell / 등자기위도')
ax.set_ylabel('He+ density (cm$^{-3}$) / He+ 밀도')
ax.set_title('Equatorial He+ profile / 적도면 He+ 프로파일')
ax.grid(True, which='both', alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Compute optical depth as a function of impact parameter.
SIGMA_HE_30P4 = 8.0e-13  # cm^2, line-center cross section for He+ 30.4 nm
G_FACTOR_S = 5.0e-7      # s^-1, resonant scattering rate at solar minimum

x_grid = np.linspace(0.5, 8.0, 80)
column_density = np.array([line_integral(model.density, x) for x in x_grid])
tau = SIGMA_HE_30P4 * column_density

# Brightness in Rayleighs: I = (g / 4 pi) * N * 1e-6 (Rayleigh definition)
brightness_R = (G_FACTOR_S / (4.0 * np.pi)) * column_density * 1.0e-6

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].plot(x_grid, tau, color='C2', lw=2)
axes[0].axhline(1.0, color='red', ls='--', label='Optically thick boundary')
axes[0].axhline(0.01, color='gray', ls=':',
                label=r'$\tau = 0.01$ (very thin)')
axes[0].set_xlabel('Impact parameter (R$_E$) / 충돌 매개변수')
axes[0].set_ylabel(r'Optical depth $\tau$ / 광학 두께')
axes[0].set_yscale('log')
axes[0].set_title('Optical depth at 30.4 nm / 30.4 nm 광학 두께')
axes[0].grid(True, which='both', alpha=0.3)
axes[0].legend()

axes[1].plot(x_grid, brightness_R, color='C3', lw=2)
axes[1].set_xlabel('Impact parameter (R$_E$) / 충돌 매개변수')
axes[1].set_ylabel('Brightness (R) / 강도')
axes[1].set_title('Predicted 30.4 nm brightness / 예측 강도')
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

tau_max = tau.max()
B_peak = brightness_R.max()
print(f'Maximum optical depth tau_max = {tau_max:.2e}')
print(f'Peak brightness               = {B_peak:.2f} R')
print(f'Paper requirement (Sec.2 #1): plasmasphere max ~10 R --> match? '
      f'{abs(B_peak - 10.0) < 5.0}')
print(f'Optically thin verified       : tau_max << 1 --> {tau_max < 1e-2}')

**Interpretation / 해석**: $\tau_\max$ remains well below 0.01 even through the densest plasmaspheric core, validating the linearity assumption $I \propto N_{\mathrm{He^+}}$. The peak brightness reproduces the paper's stated maximum of ~10 R (Section 2, requirement #1). / 가장 조밀한 플라스마권 코어를 통과해도 $\tau_\max < 0.01$로 광학적 얇음이 검증되며, 피크 강도가 §2 요구 #1의 ~10 R과 일치한다.

## Part 2: Three-Head 30° FOV Mosaic and Earth-Limb Registration / 3-헤드 30° FOV 모자이크와 지구 limb 정합

The three EUV sensor heads are tilted by 27° relative to one another so that their 30°-FOV cones overlap by 3° at the seams. We model the instantaneous 84°×30° fan in spacecraft body coordinates, sweep it through 360° spin to produce the 84°×360° annulus, and then overlay Earth's apparent limb at apogee (~7 R_E altitude → 1 R_E disk subtending ~16°).

EUV의 세 센서 헤드는 30° 시야가 이음매에서 3°씩 중첩되도록 27°씩 기울어져 순간 84°×30° 부채꼴을 형성한다. 자전을 통해 84°×360° 환형 영역을 만들고, 원지점에서 지구 limb를 정합한다.

In [ ]:
@dataclass
class EuvHeadGeometry:
    """Geometry of one EUV sensor head.

    Attributes:
        center_elev_deg: Elevation of head boresight relative to spin equator.
        fov_diameter_deg: Full conical FOV diameter.
    """

    center_elev_deg: float
    fov_diameter_deg: float = 30.0

    def in_field(self, elev_deg: np.ndarray, az_deg: np.ndarray,
                 fan_az_deg: float = 0.0) -> np.ndarray:
        """Test whether (elev, az) lies inside this head's circular FOV.

        Args:
            elev_deg: Elevation grid in degrees.
            az_deg: Azimuth grid in degrees.
            fan_az_deg: Instantaneous azimuth of the fan center.

        Returns:
            Boolean array of identical shape: True inside FOV.
        """
        delta_az = (az_deg - fan_az_deg + 180.0) % 360.0 - 180.0
        delta_el = elev_deg - self.center_elev_deg
        return np.hypot(delta_az, delta_el) <= self.fov_diameter_deg / 2.0


# Three heads tilted by 27 deg each: -27, 0, +27 elevation centers
heads = [EuvHeadGeometry(center_elev_deg=-27.0),
         EuvHeadGeometry(center_elev_deg=0.0),
         EuvHeadGeometry(center_elev_deg=+27.0)]

elev = np.linspace(-60.0, 60.0, 360)
az = np.linspace(0.0, 360.0, 720)
EL, AZ = np.meshgrid(elev, az, indexing='ij')

# Instantaneous 84 deg x 30 deg fan (all three heads, fan_az = 0).
fan_mask = np.zeros_like(EL, dtype=bool)
for h in heads:
    fan_mask |= h.in_field(EL, AZ, fan_az_deg=0.0)

# Spin-swept 84 deg x 360 deg annulus.
annulus_mask = np.zeros_like(EL, dtype=bool)
for fan_az in np.arange(0.0, 360.0, 5.0):
    for h in heads:
        annulus_mask |= h.in_field(EL, AZ, fan_az_deg=fan_az)

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
axes[0].pcolormesh(AZ, EL, fan_mask.astype(float), cmap='Blues',
                   shading='auto')
axes[0].set_title('Instantaneous fan (84$^\\circ$×30$^\\circ$)\n순간 부채꼴')
axes[0].set_xlabel('Azimuth (deg) / 방위각')
axes[0].set_ylabel('Elevation (deg) / 고도각')
axes[0].grid(True, alpha=0.3)

axes[1].pcolormesh(AZ, EL, annulus_mask.astype(float), cmap='Blues',
                   shading='auto')
axes[1].set_title('Spin-swept annulus (84$^\\circ$×360$^\\circ$)\n자전 환형 영역')
axes[1].set_xlabel('Azimuth (deg) / 방위각')
axes[1].set_ylabel('Elevation (deg) / 고도각')
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

instantaneous_coverage = fan_mask.mean() * 100.0
annulus_coverage = annulus_mask.mean() * 100.0
print(f'Instantaneous sky coverage: {instantaneous_coverage:.1f}% '
      f'(theoretical 84*30/(180*360) = {84*30/(180*360)*100:.1f}%)')
print(f'Spin-swept sky coverage   : {annulus_coverage:.1f}% '
      f'(theoretical 84*360/(180*360) = {84*360/(180*360)*100:.1f}%)')

In [ ]:
def earth_limb_overlay(spacecraft_alt_re: float = 7.0,
                       limb_az_deg: float = 180.0,
                       n_pts: int = 200) -> tuple[np.ndarray, np.ndarray]:
    """Generate Earth-limb (az, el) coordinates as seen from IMAGE apogee.

    Args:
        spacecraft_alt_re: Geocentric distance of IMAGE in R_E.
        limb_az_deg: Azimuth where the Earth disk center sits.
        n_pts: Number of points around the limb.

    Returns:
        Tuple of (az_array_deg, el_array_deg).
    """
    half_angle = np.rad2deg(np.arcsin(1.0 / spacecraft_alt_re))
    theta = np.linspace(0.0, 2.0 * np.pi, n_pts)
    az = limb_az_deg + half_angle * np.cos(theta)
    el = half_angle * np.sin(theta)
    return az, el


limb_az, limb_el = earth_limb_overlay()

fig, ax = plt.subplots(figsize=(11, 5.5))
ax.pcolormesh(AZ, EL, annulus_mask.astype(float), cmap='Blues',
              shading='auto', alpha=0.5)
ax.plot(limb_az, limb_el, color='black', lw=2.0,
        label='Earth limb at 7 R$_E$ apogee')
ax.fill(limb_az, limb_el, color='lightgreen', alpha=0.4)
ax.scatter([180.0], [0.0], color='green', s=30, label='Earth center')
ax.set_title('EUV sky coverage with Earth registered\nEUV 시야와 지구 정합')
ax.set_xlabel('Azimuth (deg) / 방위각')
ax.set_ylabel('Elevation (deg) / 고도각')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_xlim(120, 240)
ax.set_ylim(-50, 50)
plt.tight_layout()
plt.show()

earth_half_angle = np.rad2deg(np.arcsin(1.0 / 7.0))
print(f'Earth half-angle at 7 R_E apogee: {earth_half_angle:.2f} deg '
      f'(disk diameter ~{2*earth_half_angle:.1f} deg)')
print(f'Plasmasphere half-angle (4.5 R_E): '
      f'{np.rad2deg(np.arcsin(4.5 / 7.0)):.2f} deg')

**Interpretation / 해석**: The 84°×360° annulus easily contains the Earth disk (~16° diameter at 7 R_E) and the plasmapause (~80° diameter). The 0.6° resolution element on the 84°×360° annulus corresponds to 84/0.6 × 360/0.6 = 140 × 600 = 84,000 spatial cells — broadly consistent with the 52×600 skymap product after vignetting trim. / 84°×360° 환형은 ~16° 지구 원반과 ~80° 플라스마포즈를 모두 포함한다. 0.6° 해상 셀은 ~84,000 공간 셀로, 52×600 스카이맵 산물과 비네팅 트림 후 일치한다.

## Part 3: Storm-Time Plasmaspheric Plume from Polar Perspective / 극지 시점에서의 폭풍기 플라스마권 plume

When IMAGE views the plasmasphere from high northern latitude, an asymmetric storm-time He+ distribution exhibiting a sunward duskside plume becomes visible as a brightness asymmetry. We synthesize a 2-D equatorial He+ density (R, MLT) with an attached plume on the duskside, project to the sky, and forward-model the Rayleigh image.

IMAGE가 극지 시점에서 플라스마권을 볼 때, 황혼측 plume이 붙은 비대칭 폭풍기 He+ 분포가 강도 비대칭으로 나타난다. 듀스크사이드 plume이 부착된 2-D 적도면 He+ 밀도 (R, MLT)를 합성하고 강도 영상으로 forward-modeling한다.

In [ ]:
@dataclass
class PlumePlasmasphere:
    """Storm-time plasmasphere with a sunward duskside plume.

    Attributes:
        background: Underlying axisymmetric model.
        plume_mlt_deg: Magnetic-local-time center of the plume (deg from noon).
        plume_width_deg: 1-sigma half-width of the plume in MLT (deg).
        plume_amplitude: Density boost factor relative to plasmatrough.
        plume_outer_re: Outer radial extent of the plume (R_E).
    """

    background: PlasmasphereModel
    plume_mlt_deg: float = 80.0  # ~ 18 MLT (duskside)
    plume_width_deg: float = 25.0
    plume_amplitude: float = 80.0
    plume_outer_re: float = 7.5

    def density_2d(self, x_re: np.ndarray, y_re: np.ndarray) -> np.ndarray:
        """He+ density on the equatorial plane.

        Args:
            x_re: Solar-magnetospheric x in R_E (Sun direction).
            y_re: Solar-magnetospheric y in R_E (dawn-to-dusk direction).

        Returns:
            He+ density (cm^-3) on the (x, y) grid.
        """
        L = np.sqrt(x_re ** 2 + y_re ** 2)
        n_axis = self.background.density(L)
        # Plume azimuth measured from +x axis (Sun) toward +y (dusk for IMAGE)
        phi = np.rad2deg(np.arctan2(y_re, x_re))
        delta_phi = (phi - self.plume_mlt_deg + 180.0) % 360.0 - 180.0
        gauss = np.exp(-0.5 * (delta_phi / self.plume_width_deg) ** 2)
        radial_in = (L >= self.background.L_pp - 0.3)
        radial_out = (L <= self.plume_outer_re)
        plume_n = (self.plume_amplitude * gauss
                   * radial_in.astype(float)
                   * radial_out.astype(float))
        return n_axis + plume_n


plume_model = PlumePlasmasphere(background=model)
x = np.linspace(-8.0, 8.0, 240)
y = np.linspace(-8.0, 8.0, 240)
X, Y = np.meshgrid(x, y, indexing='xy')
n2d = plume_model.density_2d(X, Y)

fig, ax = plt.subplots(figsize=(7.5, 6.5))
im = ax.pcolormesh(X, Y, np.log10(np.maximum(n2d, 1e-2)),
                   cmap='magma', shading='auto', vmin=-1, vmax=2.5)
ax.add_artist(plt.Circle((0, 0), 1, color='white', alpha=0.9))
ax.text(0, 0, 'Earth', ha='center', va='center', fontsize=8)
ax.set_aspect('equal')
ax.set_xlabel('X (R$_E$, sunward) / 태양 방향')
ax.set_ylabel('Y (R$_E$, dawn→dusk) / 새벽→황혼')
ax.set_title('Storm-time He+ density (log$_{10}$ cm$^{-3}$)\n'
             '폭풍기 He+ 밀도')
plt.colorbar(im, ax=ax, label='log$_{10}$ n$_{He+}$')
plt.tight_layout()
plt.show()

In [ ]:
def project_polar_image(density_2d_fn, sc_alt_re: float = 7.0,
                        sc_lat_deg: float = 60.0, n_pix: int = 96,
                        s_steps: int = 250,
                        s_max_re: float = 14.0) -> np.ndarray:
    """Forward model EUV brightness from a high-latitude vantage point.

    Lines of sight stem from a spacecraft at (0, 0, sc_alt_re*sin(lat)) and
    point through equatorial pixels in (X, Y). The integration is done
    along straight lines in 3-D from the spacecraft toward the equatorial
    target, sampling a thin slab around z = 0 (the He+ pancake).

    Args:
        density_2d_fn: Function f(x_re, y_re) returning equatorial density.
        sc_alt_re: Geocentric distance of IMAGE in R_E.
        sc_lat_deg: Magnetic latitude of IMAGE (deg).
        n_pix: Pixels per side of the synthesized image.
        s_steps: Number of integration samples per line of sight.
        s_max_re: Half path length in R_E.

    Returns:
        Brightness image in Rayleighs (n_pix, n_pix).
    """
    sc_x = 0.0
    sc_y = 0.0
    sc_z = sc_alt_re * np.sin(np.deg2rad(sc_lat_deg))
    target_x = np.linspace(-8.0, 8.0, n_pix)
    target_y = np.linspace(-8.0, 8.0, n_pix)
    img = np.zeros((n_pix, n_pix))
    s = np.linspace(-s_max_re, s_max_re, s_steps)
    ds_cm = (s[1] - s[0]) * RE_TO_CM
    for ix, tx in enumerate(target_x):
        for iy, ty in enumerate(target_y):
            dx = tx - sc_x
            dy = ty - sc_y
            dz = 0.0 - sc_z
            length = np.sqrt(dx * dx + dy * dy + dz * dz)
            ux, uy, uz = dx / length, dy / length, dz / length
            xs = sc_x + s * ux + tx  # parametrize symmetric around target
            ys = sc_y + s * uy + ty
            zs = sc_z + s * uz
            # Pancake approximation: density rapidly falls off-equator
            scale_re = 0.6
            n_eq = density_2d_fn(xs, ys)
            n = n_eq * np.exp(-(zs / scale_re) ** 2)
            column = float(np.sum(n) * ds_cm)
            img[iy, ix] = (G_FACTOR_S / (4.0 * np.pi)) * column * 1.0e-6
    return img


image_R = project_polar_image(plume_model.density_2d, n_pix=72, s_steps=180)

fig, ax = plt.subplots(figsize=(7.5, 6.5))
im = ax.imshow(image_R, origin='lower', extent=[-8, 8, -8, 8],
               cmap='inferno')
ax.add_artist(plt.Circle((0, 0), 1, color='white', alpha=0.6))
ax.text(0, 0, 'Earth', ha='center', va='center', fontsize=8, color='black')
ax.set_xlabel('X (R$_E$, sunward) / 태양 방향')
ax.set_ylabel('Y (R$_E$, dawn→dusk) / 새벽→황혼')
ax.set_title('EUV-style polar view: storm-time plume\n'
             'EUV 극지 시점: 폭풍기 plume')
plt.colorbar(im, ax=ax, label='Brightness (R) / 강도')
plt.tight_layout()
plt.show()

print(f'Peak image brightness: {image_R.max():.2f} R')
print(f'Plume signature visible: '
      f'{image_R[image_R.shape[0]//2, image_R.shape[1]*3//4] > 1.0}')

**Interpretation / 해석**: The polar viewpoint reveals the duskside plume as an asymmetric brightness extension — exactly the signature later confirmed observationally by Burch et al. (2001). Brightness scales as expected: a ~10 R bright core, a ~5 R plume, and a dark plasmatrough beyond. / 극지 시점은 황혼측 plume을 비대칭 강도 확장으로 드러내며, 이는 Burch 외 (2001)에서 관측으로 확인된 특징이다. 코어 ~10 R, plume ~5 R, 외부 plasmatrough 어둠으로 강도가 적절히 분포한다.

## Part 4: Toy Plasmaspheric Tomography / 장난감 플라스마권 단층촬영

Because the plasmasphere is optically thin, each pixel's brightness is a *line integral* of He+ density. With many lines of sight from a moving IMAGE spacecraft, we can in principle invert the integrals to recover a 2-D density map — exactly the approach later used by Sandel et al. (2003) and Goldstein et al. (2003). We build a small linear system $A \mathbf{n} = \mathbf{b}$ where rows are sight-line path-length matrices and solve via non-negative least squares (NNLS).

광학적 얇음 덕분에 각 픽셀 강도는 He+ 밀도의 *시선 적분*이다. 이동하는 IMAGE 위성이 제공하는 다수의 시선 정보로부터 적분을 역변환해 2-D 밀도를 복원할 수 있다. 작은 선형 시스템 $A \mathbf{n} = \mathbf{b}$를 만들어 NNLS로 푼다.

In [ ]:
def build_tomography_system(true_density_grid: np.ndarray,
                            grid_extent_re: float,
                            n_views: int = 6,
                            n_rays_per_view: int = 28,
                            ) -> tuple[np.ndarray, np.ndarray]:
    """Build the line-of-sight forward operator and synthetic measurements.

    The 2-D scene is discretized on an N x N voxel grid covering
    [-extent, +extent]^2 R_E. For each viewpoint (uniformly distributed on a
    7 R_E orbit), we cast n_rays_per_view parallel lines through the grid;
    each ray contributes one row to A.

    Args:
        true_density_grid: Ground-truth 2-D density map (cm^-3).
        grid_extent_re: Half-width of the grid (R_E).
        n_views: Number of distinct viewpoints around Earth.
        n_rays_per_view: Number of parallel rays per view.

    Returns:
        Tuple of (A, b) where A has shape (n_views*n_rays_per_view, N*N) and
        b is the simulated brightness vector in Rayleigh.
    """
    n_grid = true_density_grid.shape[0]
    cell_size_re = 2.0 * grid_extent_re / n_grid
    cell_size_cm = cell_size_re * RE_TO_CM
    coords_re = np.linspace(-grid_extent_re + cell_size_re / 2,
                            grid_extent_re - cell_size_re / 2, n_grid)
    XX, YY = np.meshgrid(coords_re, coords_re, indexing='xy')

    rows = []
    bvals = []
    for view_phi in np.linspace(0.0, np.pi, n_views, endpoint=False):
        # Parallel beam direction (cos phi, sin phi).
        ux, uy = np.cos(view_phi), np.sin(view_phi)
        # Perpendicular axis along which ray impact parameters spread.
        px, py = -np.sin(view_phi), np.cos(view_phi)
        impacts = np.linspace(-grid_extent_re * 0.95,
                              grid_extent_re * 0.95, n_rays_per_view)
        for b_imp in impacts:
            # Soft line-of-sight kernel: gaussian half-width ~ cell_size.
            dist_perp = (XX - b_imp * px) * px + (YY - b_imp * py) * py
            kernel = np.exp(-0.5 * (dist_perp / (cell_size_re * 0.7)) ** 2)
            kernel = kernel / kernel.sum() * n_grid  # normalize per ray
            # Path length through one cell ~ cell_size_cm; effective integral
            row = (kernel * cell_size_cm).flatten()
            measurement = (G_FACTOR_S / (4.0 * np.pi)) * 1.0e-6 * (
                row @ true_density_grid.flatten())
            rows.append(row * (G_FACTOR_S / (4.0 * np.pi)) * 1.0e-6)
            bvals.append(measurement)
    return np.array(rows), np.array(bvals)


def reconstruct_tomography(A: np.ndarray, b: np.ndarray,
                            n_grid: int) -> np.ndarray:
    """Recover the density map via non-negative least squares.

    Args:
        A: Forward operator (n_meas, n_grid*n_grid).
        b: Brightness measurements (n_meas,).
        n_grid: Side length of the square reconstruction grid.

    Returns:
        Reconstructed density map (cm^-3) of shape (n_grid, n_grid).
    """
    n_recovered, _ = nnls(A, b, maxiter=2 * A.shape[1])
    return n_recovered.reshape(n_grid, n_grid)


n_grid = 30
extent_re = 6.0
coords = np.linspace(-extent_re, extent_re, n_grid)
XG, YG = np.meshgrid(coords, coords, indexing='xy')
true_density = plume_model.density_2d(XG, YG)

A, b = build_tomography_system(true_density, extent_re,
                                n_views=8, n_rays_per_view=30)
noise = rng.normal(0.0, 0.05 * b.std(), b.shape)  # 5% measurement noise
b_noisy = np.maximum(b + noise, 0.0)
recon = reconstruct_tomography(A, b_noisy, n_grid)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
vmax = max(true_density.max(), recon.max())
im0 = axes[0].pcolormesh(XG, YG, true_density, cmap='magma',
                          shading='auto', vmin=0, vmax=vmax)
axes[0].set_title('Ground truth / 진짜 밀도')
axes[0].set_aspect('equal')
plt.colorbar(im0, ax=axes[0], label='cm$^{-3}$')

im1 = axes[1].pcolormesh(XG, YG, recon, cmap='magma',
                          shading='auto', vmin=0, vmax=vmax)
axes[1].set_title('NNLS reconstruction\nNNLS 복원')
axes[1].set_aspect('equal')
plt.colorbar(im1, ax=axes[1], label='cm$^{-3}$')

residual = recon - true_density
im2 = axes[2].pcolormesh(XG, YG, residual, cmap='RdBu_r', shading='auto',
                          vmin=-vmax / 2, vmax=vmax / 2)
axes[2].set_title('Residual / 잔차')
axes[2].set_aspect('equal')
plt.colorbar(im2, ax=axes[2], label='cm$^{-3}$')

for ax in axes:
    ax.add_artist(plt.Circle((0, 0), 1, color='white', alpha=0.7))
    ax.set_xlabel('X (R$_E$)')
    ax.set_ylabel('Y (R$_E$)')
plt.tight_layout()
plt.show()

rmse = np.sqrt(np.mean((recon - true_density) ** 2))
rel = rmse / true_density.max()
print(f'Reconstruction RMSE        : {rmse:.2f} cm^-3')
print(f'Relative to peak density   : {rel*100:.1f}%')
print(f'Number of measurements     : {A.shape[0]}')
print(f'Number of unknown voxels   : {A.shape[1]}')
print(f'Underdetermination factor  : {A.shape[1]/A.shape[0]:.1f}x')

**Interpretation / 해석**: With 8 viewpoints × 30 rays = 240 measurements over 900 voxels (~3.75× underdetermined), NNLS still recovers the dominant plasmasphere core and the duskside plume; residuals concentrate near the plasmapause edge where the gradient is steepest. In practice, IMAGE accumulated thousands of skymap cells over each orbit, making the inverse problem far better conditioned. This toy demonstration validates the *principle* underlying real plasmaspheric tomography (Sandel et al. 2003; Goldstein et al. 2003). / 8 시점 × 30 광선 = 240 측정값으로 900 voxel을 (~3.75배 미결정) NNLS로 복원해도 코어와 황혼측 plume을 회수한다. 실제 IMAGE는 매 궤도 수천 개 스카이맵 셀을 누적하여 역문제가 훨씬 잘 정의된다. 본 장난감 시연은 실제 플라스마권 단층촬영(Sandel 외 2003; Goldstein 외 2003)의 *원리*를 검증한다.

## Summary / 요약

| Concept / 개념 | This Paper / 이 논문 | Modern Equivalent / 현대 동등물 |
|---|---|---|
| Optical thinness at 30.4 nm / 30.4 nm 광학 얇음 | Linear $I \propto N_{\mathrm{He^+}}$ inversion (no RT) | Same principle; modern global He+ retrievals via plasmaspheric tomography |
| Three-head FOV mosaic / 3-헤드 FOV 모자이크 | 3 × 30° heads tilted by 27° → 84°×30° fan + spin → 84°×360° annulus | TWINS LAD (two-spacecraft stereo); SMILE-SXI mosaic of soft X-ray cusp |
| Spin-scan TDI skymap / 자전 TDI 스카이맵 | 52 × 600 cell on-board mapping with five lookup tables | CCD time-domain integration; PIXIE-style coadd; modern FPGA-based on-board pipelines |
| Storm-time plume imaging / 폭풍기 plume 영상화 | Predicted; observed shortly after launch (Burch et al. 2001) | Continuous monitoring via Goldstein et al. dataset; assimilation in IFM/SAMI3 |
| Plasmaspheric tomography / 플라스마권 단층촬영 | Forward model invertible due to optical thinness | Sandel & Denton (2007), Galvan et al. (2008) tomographic retrievals |

**Connections back to the paper / 논문과의 연결**:
- **Part 1** verifies the §2 optical-thinness assumption that justifies the entire EUV concept and reproduces the ~10 R peak brightness requirement.
- **Part 2** reproduces the §3.1 instrument geometry (84°×30° fan, 84°×360° annulus, three-head 27° tilt, 3° overlap) and locates the Earth limb at IMAGE apogee.
- **Part 3** simulates the polar viewpoint signature (storm-time duskside plume) that motivated EUV's existence and that Burch et al. (2001) confirmed observationally only months after IMAGE's March 2000 launch.
- **Part 4** demonstrates the *invertibility* enabled by optical thinness — a property that subsequent IMAGE/EUV science (plasmaspheric tomography of Sandel, Goldstein, Galvan and others) depended upon.